In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget -q https://raw.githubusercontent.com/eddelbuettel/r2u/master/inst/scripts/add_cranapt_jammy.sh -O add_cranapt.sh
!bash add_cranapt.sh
!apt-get install -y r-bioc-diffbind r-bioc-deseq2 --quiet

In [ ]:
!apt-get install -y bedtools --quiet
!pip install pybedtools --quiet

In [ ]:
import subprocess

result = subprocess.run(['Rscript', '-e', 'cat("DiffBind:", "DiffBind" %in% rownames(installed.packages()), "\\n"); cat("DESeq2:", "DESeq2" %in% rownames(installed.packages()), "\\n")'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import subprocess

r_script = '''
library(DiffBind)

bam_dir  <- "/content/drive/Shareddrives/Chd8/ChIP_ATAC_BAM"
peak_dir <- "/content/drive/Shareddrives/Chd8/MACS2_peaks/ATAC"

samples <- data.frame(
  SampleID  = c("WT_rep1", "WT_rep2", "KO1_rep1", "KO1_rep2", "KO2_rep1", "KO2_rep2"),
  Condition = c("WT", "WT", "KO", "KO", "KO", "KO"),
  Replicate = c(1, 2, 1, 2, 1, 2),
  bamReads  = c(
    file.path(bam_dir, "ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_1_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_1_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_2_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_2_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam")
  ),
  Peaks = c(
    file.path(peak_dir, "ATAC_Diff_1/ATAC_Diff_1_peaks.narrowPeak"),
    file.path(peak_dir, "ATAC_Diff_2/ATAC_Diff_2_peaks.narrowPeak"),
    file.path(peak_dir, "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1_peaks.narrowPeak"),
    file.path(peak_dir, "KO_1_ATAC_Diff_2/KO_1_ATAC_Diff_2_peaks.narrowPeak"),
    file.path(peak_dir, "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1_peaks.narrowPeak"),
    file.path(peak_dir, "KO_2_ATAC_Diff_2/KO_2_ATAC_Diff_2_peaks.narrowPeak")
  ),
  PeakCaller = "narrow"
)

atac <- dba(sampleSheet = samples)
print(atac)

consensus <- dba.peakset(atac, bRetrieve = TRUE)
df <- as.data.frame(consensus)[, c("seqnames", "start", "end")]
write.table(df, "/content/consensus_peaks.bed", sep = "\\t",
            quote = FALSE, row.names = FALSE, col.names = FALSE)
cat("\\nWrote", nrow(df), "consensus peaks to BED\\n")
'''
with open('/content/step_consensus.R', 'w') as f:
    f.write(r_script)

process = subprocess.Popen(['Rscript', '/content/step_consensus.R'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

In [ ]:
import subprocess
import os
import pandas as pd

bam_dir  = "/content/drive/Shareddrives/Chd8/ChIP_ATAC_BAM"
peak_dir = "/content/drive/Shareddrives/Chd8/MACS2_peaks/ATAC"
tss_path = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"

# The 5 samples retained in the final (balanced, outlier-excluded) analysis
samples = {
    "WT_rep1":  {
        "bam":  f"{bam_dir}/ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam",
        "peak": f"{peak_dir}/ATAC_Diff_1/ATAC_Diff_1_peaks.narrowPeak",
    },
    "WT_rep2":  {
        "bam":  f"{bam_dir}/ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam",
        "peak": f"{peak_dir}/ATAC_Diff_2/ATAC_Diff_2_peaks.narrowPeak",
    },
    "KO1_rep2": {
        "bam":  f"{bam_dir}/KO_1_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam",
        "peak": f"{peak_dir}/KO_1_ATAC_Diff_2/KO_1_ATAC_Diff_2_peaks.narrowPeak",
    },
    "KO2_rep1": {
        "bam":  f"{bam_dir}/KO_2_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam",
        "peak": f"{peak_dir}/KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1_peaks.narrowPeak",
    },
    "KO2_rep2": {
        "bam":  f"{bam_dir}/KO_2_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam",
        "peak": f"{peak_dir}/KO_2_ATAC_Diff_2/KO_2_ATAC_Diff_2_peaks.narrowPeak",
    },
}
# NOTE: KO1_rep1 deliberately excluded — confirmed PCA outlier, excluded from
# all downstream analyses including this QC step, for consistency.

os.makedirs('/content/qc_frip', exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
# 1) Install deepTools if not already present
# ──────────────────────────────────────────────────────────────────────────────
subprocess.run(['pip', 'install', 'deeptools', '--quiet'], capture_output=True)

# ──────────────────────────────────────────────────────────────────────────────
# 2) Ensure BAMs are indexed (required by deepTools)
# ──────────────────────────────────────────────────────────────────────────────
for name, paths in samples.items():
    bai = paths['bam'] + '.bai'
    if not os.path.exists(bai):
        print(f"Indexing {name}...")
        subprocess.run(['samtools', 'index', paths['bam']], capture_output=True)

# ──────────────────────────────────────────────────────────────────────────────
# 3) Sort/clean each sample's peak file into a plain 3-col BED (plotEnrichment
#    just needs coordinates) — MACS2 narrowPeak has extra columns, which is
#    fine, but we make a clean version to avoid parsing issues.
# ──────────────────────────────────────────────────────────────────────────────
clean_peak_paths = {}
for name, paths in samples.items():
    clean_path = f'/content/qc_frip/{name}_peaks_clean.bed'
    subprocess.run(
        f"cut -f1-3 {paths['peak']} | sort -k1,1 -k2,2n > {clean_path}",
        shell=True, capture_output=True
    )
    clean_peak_paths[name] = clean_path

# ──────────────────────────────────────────────────────────────────────────────
# 4) Run deepTools plotEnrichment per sample: peaks region set + TSS region
#    set together, with --outRawCounts to get exact numeric fractions
#    (not just a plot).
# ──────────────────────────────────────────────────────────────────────────────
results = []

for name, paths in samples.items():
    print(f"\nProcessing {name}...")
    out_raw = f'/content/qc_frip/{name}_enrichment_raw.tsv'
    out_png = f'/content/qc_frip/{name}_enrichment.png'

    cmd = [
        'plotEnrichment',
        '--bamfiles', paths['bam'],
        '--BED', clean_peak_paths[name], tss_path,
        '--labels', name,                          # 1 label for 1 BAM file
        '--regionLabels', 'MACS2_peaks', 'TSS_+/-2kb',  # 2 labels for 2 BED files
        '-o', out_png,
        '--outRawCounts', out_raw,
        '-p', '2',
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  ERROR: {proc.stderr[-1500:]}")
        continue

    raw = pd.read_csv(out_raw, sep='\t')
    print(raw.to_string(index=False))

    frip_row = raw[raw['featureType'] == 'MACS2_peaks']
    tss_row  = raw[raw['featureType'] == 'TSS_+/-2kb']

    frip_val = frip_row['percent'].values[0] if len(frip_row) else None
    tss_val  = tss_row['percent'].values[0] if len(tss_row) else None

    results.append({
        'Sample': name,
        'FRiP_percent': frip_val,
        'Fraction_reads_at_TSS_percent': tss_val,
    })

# ──────────────────────────────────────────────────────────────────────────────
# 5) Summary table + ENCODE threshold check
#    ENCODE's commonly-cited ATAC-seq FRiP guideline: > 0.2 (20%) is
#    considered acceptable; > 0.3 is good. Cite this threshold, don't
#    invent a stricter one.
# ──────────────────────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("SUMMARY — FRiP and fraction of reads at TSS, per sample")
print("=" * 70)
print(summary_df.to_string(index=False))

if not summary_df.empty:
    summary_df['FRiP_frac'] = summary_df['FRiP_percent'] / 100
    passed = (summary_df['FRiP_frac'] > 0.20).sum()
    print(f"\nSamples with FRiP > 0.20 (ENCODE minimum guideline): {passed}/{len(summary_df)}")
    for _, row in summary_df.iterrows():
        status = "PASS" if row['FRiP_frac'] > 0.20 else "BELOW 0.20 — flag for review"
        print(f"  {row['Sample']}: FRiP = {row['FRiP_frac']:.3f}  [{status}]")

out_csv = '/content/qc_frip/FRiP_TSS_summary.csv'
summary_df.to_csv(out_csv, index=False)
print(f"\n✓ Saved: {out_csv}")

print("\n" + "-" * 70)
print("IMPORTANT: 'Fraction_reads_at_TSS_percent' above is a proxy metric,")
print("not the formal ENCODE TSS ENRICHMENT SCORE (which normalizes signal")
print("at the TSS to flanking background, not just overlap fraction). If a")
print("reviewer specifically expects the formal ENCODE score, that requires")
print("an additional calculation (e.g., via the `ataqv` tool or ENCODE's own")
print("qc pipeline) — let me know if you want that script too.")
print("-" * 70)

from google.colab import files
files.download(out_csv)

In [ ]:
import os
import subprocess

bam_dir = '/content/drive/Shareddrives/Chd8/ChIP_ATAC_BAM'
samples = {
    'WT_rep1':  f'{bam_dir}/ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam',
    'WT_rep2':  f'{bam_dir}/ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam',
    'KO1_rep1': f'{bam_dir}/KO_1_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam',
    'KO1_rep2': f'{bam_dir}/KO_1_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam',
    'KO2_rep1': f'{bam_dir}/KO_2_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam',
    'KO2_rep2': f'{bam_dir}/KO_2_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam',
}

os.makedirs('/content/frag_bams', exist_ok=True)

# Make sure samtools is available (r2u install may not have pulled it in)
subprocess.run(['apt-get', 'install', '-y', 'samtools', '--quiet'], capture_output=True)

for name, path in samples.items():
    out_bam = f'/content/frag_bams/{name}.frag.bam'
    print(f"Processing {name}...")
    # -f 0x40: first-in-pair only (one record per fragment, avoids double-counting)
    # -F 0x904: exclude unmapped, secondary, duplicate reads
    cmd = f'samtools view -@ 2 -b -f 0x40 -F 0x904 "{path}" > "{out_bam}" && samtools index "{out_bam}"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("  ERROR:", result.stderr[-1000:])
    else:
        size_mb = os.path.getsize(out_bam) / (1024 * 1024)
        print(f"  Done → {out_bam} ({size_mb:.1f} MB)")

In [ ]:
import subprocess

subprocess.run(['apt-get', 'install', '-y', 'bedtools', '--quiet'], capture_output=True)

frag_bams = [
    '/content/frag_bams/WT_rep1.frag.bam',
    '/content/frag_bams/WT_rep2.frag.bam',
    '/content/frag_bams/KO1_rep1.frag.bam',
    '/content/frag_bams/KO1_rep2.frag.bam',
    '/content/frag_bams/KO2_rep1.frag.bam',
    '/content/frag_bams/KO2_rep2.frag.bam',
]

cmd = ['bedtools', 'multicov', '-bams', *frag_bams, '-bed', '/content/consensus_peaks.bed']
with open('/content/atac_counts.txt', 'w') as out:
    process = subprocess.Popen(cmd, stdout=out, stderr=subprocess.PIPE, text=True)
    _, err = process.communicate()

print("Exit code:", process.returncode)
if err:
    print("STDERR:", err[-2000:])

import os
if os.path.exists('/content/atac_counts.txt'):
    print("Output size:", os.path.getsize('/content/atac_counts.txt'), "bytes")

In [ ]:
import pandas as pd

cols = ['chr', 'start', 'end', 'WT_rep1', 'WT_rep2', 'KO1_rep1', 'KO1_rep2', 'KO2_rep1', 'KO2_rep2']
counts = pd.read_csv('/content/atac_counts.txt', sep='\t', header=None, names=cols)

print("Shape:", counts.shape)
print("\nFirst few rows:")
print(counts.head())

print("\nPer-sample total fragment counts (library size):")
print(counts[['WT_rep1', 'WT_rep2', 'KO1_rep1', 'KO1_rep2', 'KO2_rep1', 'KO2_rep2']].sum())

print("\nRows with zero counts in ALL samples:", (counts[['WT_rep1', 'WT_rep2', 'KO1_rep1', 'KO1_rep2', 'KO2_rep1', 'KO2_rep2']].sum(axis=1) == 0).sum())

print("\nPer-sample summary stats:")
print(counts[['WT_rep1', 'WT_rep2', 'KO1_rep1', 'KO1_rep2', 'KO2_rep1', 'KO2_rep2']].describe())

counts.to_csv('/content/atac_counts_labeled.csv', index=False)

In [ ]:
import subprocess

r_script = '''
library(DESeq2)

counts <- read.csv("/content/atac_counts_labeled.csv")

count_mat <- as.matrix(counts[, c("WT_rep1","WT_rep2","KO1_rep1","KO1_rep2","KO2_rep1","KO2_rep2")])
rownames(count_mat) <- paste0(counts$chr, ":", counts$start, "-", counts$end)

coldata <- data.frame(
  condition = factor(c("WT","WT","KO","KO","KO","KO"), levels = c("WT","KO")),
  row.names = colnames(count_mat)
)

cat("Count matrix dims:", dim(count_mat), "\\n")
cat("Column data:\\n")
print(coldata)

dds <- DESeqDataSetFromMatrix(countData = count_mat, colData = coldata, design = ~ condition)
dds <- DESeq(dds)

res <- results(dds, contrast = c("condition", "KO", "WT"))
res_df <- as.data.frame(res)
res_df$region <- rownames(res_df)

cat("\\nDESeq2 summary:\\n")
summary(res)

write.csv(res_df, "/content/atac_deseq2_results.csv", row.names = FALSE)
cat("\\nSaved to /content/atac_deseq2_results.csv\\n")
'''
with open('/content/run_deseq2.R', 'w') as f:
    f.write(r_script)

process = subprocess.Popen(['Rscript', '/content/run_deseq2.R'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

In [ ]:
import subprocess

r_script = '''
library(DESeq2)

counts <- read.csv("/content/atac_counts_labeled.csv")
count_mat <- as.matrix(counts[, c("WT_rep1","WT_rep2","KO1_rep1","KO1_rep2","KO2_rep1","KO2_rep2")])
rownames(count_mat) <- paste0(counts$chr, ":", counts$start, "-", counts$end)

coldata <- data.frame(
  condition = factor(c("WT","WT","KO","KO","KO","KO"), levels = c("WT","KO")),
  row.names = colnames(count_mat)
)

dds <- DESeqDataSetFromMatrix(countData = count_mat, colData = coldata, design = ~ condition)
dds <- DESeq(dds)

norm_counts <- counts(dds, normalized = TRUE)
norm_df <- as.data.frame(norm_counts)
norm_df$region <- rownames(norm_df)

write.csv(norm_df, "/content/atac_normalized_counts.csv", row.names = FALSE)
cat("Saved normalized counts\\n")
'''
with open('/content/export_norm.R', 'w') as f:
    f.write(r_script)

process = subprocess.Popen(['Rscript', '/content/export_norm.R'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

In [ ]:
import subprocess

import subprocess
import os

r_script = '''
# ── Auto-install missing packages fast using pre-compiled binaries ──
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))

required_pkgs <- c("pheatmap", "RColorBrewer", "ggplot2")
for (pkg in required_pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, quiet = TRUE)
  }
}

library(DESeq2)
library(pheatmap)
library(RColorBrewer)

counts <- read.csv("/content/atac_counts_labeled.csv")
count_mat <- as.matrix(counts[, c("WT_rep1","WT_rep2","KO1_rep1","KO1_rep2","KO2_rep1","KO2_rep2")])
rownames(count_mat) <- paste0(counts$chr, ":", counts$start, "-", counts$end)

coldata <- data.frame(
  condition = factor(c("WT","WT","KO","KO","KO","KO"), levels = c("WT","KO")),
  clone     = factor(c("WT","WT","KO1","KO1","KO2","KO2")),
  row.names = colnames(count_mat)
)

dds <- DESeqDataSetFromMatrix(countData = count_mat, colData = coldata, design = ~ condition)
dds <- DESeq(dds)
vsd <- vst(dds, blind = TRUE)

# --- PCA ---
png("/content/pca_check.png", width = 900, height = 700, res = 120)
pca_data <- plotPCA(vsd, intgroup = c("condition", "clone"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))
library(ggplot2)
p <- ggplot(pca_data, aes(PC1, PC2, color = clone, shape = condition, label = name)) +
  geom_point(size = 4) +
  geom_text(vjust = -1, size = 3) +
  xlab(paste0("PC1: ", percentVar[1], "% variance")) +
  ylab(paste0("PC2: ", percentVar[2], "% variance")) +
  theme_bw()
print(p)
dev.off()

# --- Sample-to-sample distance heatmap / hierarchical clustering ---
sampleDists <- dist(t(assay(vsd)))
sampleDistMatrix <- as.matrix(sampleDists)

png("/content/clustering_check.png", width = 800, height = 700, res = 120)
colors <- colorRampPalette(rev(brewer.pal(9, "Blues")))(255)
pheatmap(sampleDistMatrix,
         clustering_distance_rows = sampleDists,
         clustering_distance_cols = sampleDists,
         col = colors)
dev.off()

cat("Saved /content/pca_check.png and /content/clustering_check.png\\n")

# Also print numeric PC1/PC2 coordinates for a quick look without opening the image
cat("\\nPCA coordinates:\\n")
print(pca_data[, c("name", "PC1", "PC2", "condition", "clone")])
'''
with open('/content/pca_outlier_check.R', 'w') as f:
    f.write(r_script)

process = subprocess.Popen(['Rscript', '/content/pca_outlier_check.R'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

from IPython.display import Image, display
display(Image('/content/pca_check.png'))
display(Image('/content/clustering_check.png'))

In [ ]:
import subprocess

# ---------------------------------------------------------------------------
# STEP A: Rebuild consensus peak set — balanced (1 sample per KO clone),
# and excluding KO1_rep1 entirely since PCA confirms it as an outlier/likely mislabel
# ---------------------------------------------------------------------------

r_script_consensus = '''
library(DiffBind)

bam_dir  <- "/content/drive/Shareddrives/Chd8/ChIP_ATAC_BAM"
peak_dir <- "/content/drive/Shareddrives/Chd8/MACS2_peaks/ATAC"

# Balanced set for CONSENSUS BUILDING ONLY: 2 WT + 1 rep per KO clone
# KO1_rep1 excluded entirely (confirmed outlier / likely mislabeled)
samples_balanced <- data.frame(
  SampleID  = c("WT_rep1", "WT_rep2", "KO1_rep2", "KO2_rep1"),
  Condition = c("WT", "WT", "KO", "KO"),
  Replicate = c(1, 2, 1, 1),
  bamReads  = c(
    file.path(bam_dir, "ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_1_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_2_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam")
  ),
  Peaks = c(
    file.path(peak_dir, "ATAC_Diff_1/ATAC_Diff_1_peaks.narrowPeak"),
    file.path(peak_dir, "ATAC_Diff_2/ATAC_Diff_2_peaks.narrowPeak"),
    file.path(peak_dir, "KO_1_ATAC_Diff_2/KO_1_ATAC_Diff_2_peaks.narrowPeak"),
    file.path(peak_dir, "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1_peaks.narrowPeak")
  ),
  PeakCaller = "narrow"
)

atac_balanced <- dba(sampleSheet = samples_balanced)
print(atac_balanced)

consensus <- dba.peakset(atac_balanced, bRetrieve = TRUE, minOverlap = 2)
df <- as.data.frame(consensus)[, c("seqnames", "start", "end")]
write.table(df, "/content/consensus_peaks_balanced.bed", sep = "\\t",
            quote = FALSE, row.names = FALSE, col.names = FALSE)
cat("\\nWrote", nrow(df), "balanced consensus peaks to BED (previous unbalanced set had 84939)\\n")
'''
with open('/content/step_consensus_balanced.R', 'w') as f:
    f.write(r_script_consensus)

process = subprocess.Popen(['Rscript', '/content/step_consensus_balanced.R'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()


# ---------------------------------------------------------------------------
# STEP B: Recount reads against the BALANCED consensus set, using all
# remaining samples EXCEPT KO1_rep1 (5 samples total: 2 WT + 1 KO1 + 2 KO2)
# ---------------------------------------------------------------------------

frag_bams = [
    '/content/frag_bams/WT_rep1.frag.bam',
    '/content/frag_bams/WT_rep2.frag.bam',
    # KO1_rep1 deliberately excluded — confirmed outlier / likely mislabeled
    '/content/frag_bams/KO1_rep2.frag.bam',
    '/content/frag_bams/KO2_rep1.frag.bam',
    '/content/frag_bams/KO2_rep2.frag.bam',
]

cmd = ['bedtools', 'multicov', '-bams', *frag_bams, '-bed', '/content/consensus_peaks_balanced.bed']
with open('/content/atac_counts_balanced.txt', 'w') as out:
    process = subprocess.Popen(cmd, stdout=out, stderr=subprocess.PIPE, text=True)
    _, err = process.communicate()

print("Exit code:", process.returncode)
if err:
    print("STDERR:", err[-2000:])


# ---------------------------------------------------------------------------
# STEP C: Load counts, label columns, save labeled CSV
# ---------------------------------------------------------------------------
import pandas as pd

cols = ['chr', 'start', 'end', 'WT_rep1', 'WT_rep2', 'KO1_rep2', 'KO2_rep1', 'KO2_rep2']
counts = pd.read_csv('/content/atac_counts_balanced.txt', sep='\t', header=None, names=cols)

print("Shape:", counts.shape)
print("\nPer-sample total fragment counts (library size):")
print(counts[['WT_rep1', 'WT_rep2', 'KO1_rep2', 'KO2_rep1', 'KO2_rep2']].sum())

counts.to_csv('/content/atac_counts_balanced_labeled.csv', index=False)


# ---------------------------------------------------------------------------
# STEP D: DESeq2 on the 5-sample (outlier-excluded) balanced dataset
# ---------------------------------------------------------------------------
r_script_deseq2 = '''
library(DESeq2)

counts <- read.csv("/content/atac_counts_balanced_labeled.csv")

count_mat <- as.matrix(counts[, c("WT_rep1","WT_rep2","KO1_rep2","KO2_rep1","KO2_rep2")])
rownames(count_mat) <- paste0(counts$chr, ":", counts$start, "-", counts$end)

coldata <- data.frame(
  condition = factor(c("WT","WT","KO","KO","KO"), levels = c("WT","KO")),
  clone     = factor(c("WT","WT","KO1","KO2","KO2")),
  row.names = colnames(count_mat)
)

dds <- DESeqDataSetFromMatrix(countData = count_mat, colData = coldata, design = ~ condition)
dds <- DESeq(dds)

res <- results(dds, contrast = c("condition", "KO", "WT"))
res_df <- as.data.frame(res)
res_df$region <- rownames(res_df)

cat("DESeq2 summary (balanced, outlier excluded):\\n")
summary(res)

write.csv(res_df, "/content/atac_deseq2_results_balanced.csv", row.names = FALSE)
cat("Saved to /content/atac_deseq2_results_balanced.csv\\n")

# --- Directionality on SIGNIFICANT peaks only, per Alessio's request ---
for (p in c(0.01, 0.05, 0.1)) {
  sig <- res_df[!is.na(res_df$padj) & res_df$padj < p, ]
  gained <- sum(sig$log2FoldChange > 0)
  lost   <- sum(sig$log2FoldChange < 0)
  cat(sprintf("\\npadj < %.2f: %d significant peaks | %d gained (%.1f%%) | %d lost (%.1f%%)\\n",
              p, nrow(sig), gained, 100*gained/nrow(sig), lost, 100*lost/nrow(sig)))
}
'''
with open('/content/run_deseq2_balanced.R', 'w') as f:
    f.write(r_script_deseq2)

process = subprocess.Popen(['Rscript', '/content/run_deseq2_balanced.R'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

In [ ]:
import subprocess

r_script = '''
library(DESeq2)

counts <- read.csv("/content/atac_counts_balanced_labeled.csv")
count_mat <- as.matrix(counts[, c("WT_rep1","WT_rep2","KO1_rep2","KO2_rep1","KO2_rep2")])
rownames(count_mat) <- paste0(counts$chr, ":", counts$start, "-", counts$end)

coldata <- data.frame(
  condition = factor(c("WT","WT","KO","KO","KO"), levels = c("WT","KO")),
  row.names = colnames(count_mat)
)

dds <- DESeqDataSetFromMatrix(countData = count_mat, colData = coldata, design = ~ condition)
dds <- DESeq(dds)

norm_counts <- counts(dds, normalized = TRUE)
norm_df <- as.data.frame(norm_counts)
norm_df$region <- rownames(norm_df)

write.csv(norm_df, "/content/atac_normalized_counts_balanced.csv", row.names = FALSE)
cat("Saved balanced normalized counts\\n")
'''
with open('/content/export_norm_balanced.R', 'w') as f:
    f.write(r_script)

process = subprocess.Popen(['Rscript', '/content/export_norm_balanced.R'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

In [ ]:
from google.colab import files

S11_PATH = "/content/Supplementary_Table_S11_ATAC_DESeq2_results_balanced.csv"
atac_res.to_csv(S11_PATH, index=False)
print(f"✅ S11 saved: {S11_PATH} ({len(atac_res):,} regions)")

files.download(S11_PATH)

In [ ]:
import pandas as pd
import numpy as np
import pybedtools
import os
from scipy.stats import fisher_exact

# ==========================================
# 📁 PATHS — identical to your Supp Fig 1 / triple-overlap cell
# ==========================================
BASE_DIR  = "/content/drive/MyDrive/Chd8 data"

RNA_PATH  = os.path.join(BASE_DIR, "deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
TSS_PATH  = os.path.join(BASE_DIR, "annotations/Mus_musculus_TSS_2kb_sorted.bed")
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"

# ★ Corrected, balanced, outlier-excluded ATAC-seq DESeq2 results (from Cell 12/16)
ATAC_DESEQ2_PATH = "/content/atac_deseq2_results_balanced.csv"

ATAC_LFC_THRESH  = 0.5
ATAC_PADJ_THRESH = 0.05   # same thresholds used for Fig 6C / Fig 7C / Supp Fig 1

def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ==========================================
# 1. DEGs (RNA-seq)
# ==========================================
print("📥 Loading RNA-seq...")
rna = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
rna['padj'] = rna['padj'].replace(0, 1e-300)
if 'baseMean' in rna.columns:
    rna = rna[rna['baseMean'] > 10]
rna['gene_upper'] = rna['GENESYMBOL'].astype(str).str.upper().str.strip()

degs = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'].abs() > 0.5)]['gene_upper'])
print(f"✅ DEGs: {len(degs):,}")

# ==========================================
# 2. CHD8 ChIP targets (NPC-specific, TSS ± 2kb)
# ==========================================
print("\n🔬 Loading ChIP targets...")
npc_bt   = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
esc_bt   = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
tss_bt   = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
npc_only = npc_bt.subtract(esc_bt, A=True)
hits     = tss_bt.intersect(npc_only, u=True, wa=True)
chip_targets = set([str(f[3]).upper().strip() for f in hits if len(str(f[3])) > 1])
print(f"✅ ChIP targets: {len(chip_targets):,}")

# ==========================================
# 3. Corrected ATAC-changed genes (DESeq2, balanced, outlier-excluded)
# ==========================================
print("\n🔬 Loading balanced ATAC-seq DESeq2 results...")
atac_res = pd.read_csv(ATAC_DESEQ2_PATH).dropna(subset=['log2FoldChange'])
coords = atac_res['region'].str.extract(r'(?P<chr>[^:]+):(?P<start>\d+)-(?P<end>\d+)')
atac_res['chr']   = coords['chr']
atac_res['start'] = coords['start'].astype(int)
atac_res['end']   = coords['end'].astype(int)

changed_atac = atac_res[(atac_res['padj'] < ATAC_PADJ_THRESH) &
                         (atac_res['log2FoldChange'].abs() > ATAC_LFC_THRESH)]
CHANGED_BED  = "/content/changed_atac_balanced.bed"
changed_atac[['chr', 'start', 'end']].to_csv(CHANGED_BED, sep='\t', header=False, index=False)

atac_bt_final   = pybedtools.BedTool(CHANGED_BED).each(fix_naming).sort()
atac_hits_final = tss_bt.intersect(atac_bt_final, u=True, wa=True)
atac_genes = set([str(f[3]).upper().strip() for f in atac_hits_final if len(str(f[3])) > 1])

print(f"✅ ATAC-changed regions (padj<{ATAC_PADJ_THRESH}, |LFC|>{ATAC_LFC_THRESH}): {len(changed_atac):,}")
print(f"✅ ATAC-changed genes (TSS±2kb): {len(atac_genes):,}")
print(f"   ⚠️ Sanity check — this should equal 473 (matches Supp Fig 1 / Table S12 ATAC-changed total)")

# ==========================================
# 4. Bound / non-DEG / ATAC-change comparison — corrected version of the
#    Discussion paragraph's Fisher's exact test
# ==========================================
bound_not_deg      = chip_targets - degs
bound_not_deg_atac = bound_not_deg & atac_genes
bound_deg          = chip_targets & degs
bound_deg_atac      = bound_deg & atac_genes

print(f"\nCHD8-bound, transcriptionally unaffected: {len(bound_not_deg):,}")
print(f"Of those, with corrected ATAC changes: {len(bound_not_deg_atac):,}")
print(f"Percentage: {100*len(bound_not_deg_atac)/len(bound_not_deg):.2f}%")

print(f"\nCHD8-bound, differentially expressed: {len(bound_deg):,}")
print(f"Of those, with corrected ATAC changes: {len(bound_deg_atac):,}")
print(f"Percentage: {100*len(bound_deg_atac)/len(bound_deg):.2f}%")

# Contingency table:
#                  ATAC-changed   ATAC-unchanged
# Bound + DEG            a              b
# Bound + not-DEG        c              d
a = len(bound_deg & atac_genes)
b = len(bound_deg - atac_genes)
c = len(bound_not_deg & atac_genes)
d = len(bound_not_deg - atac_genes)

odds, pval = fisher_exact([[a, b], [c, d]])
print(f"\nFisher's exact test (DEG vs non-DEG bound genes ~ corrected ATAC change):")
print(f"  Contingency: [[{a},{b}],[{c},{d}]]")
print(f"  Odds ratio: {odds:.3f}, p = {pval:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, pearsonr
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75

# ── Load BALANCED, outlier-excluded DESeq2 results and normalized counts ──────
res  = pd.read_csv('/content/atac_deseq2_results_balanced.csv')
norm = pd.read_csv('/content/atac_normalized_counts_balanced.csv')

merged = res.merge(norm, on='region')
merged = merged.dropna(subset=['log2FoldChange'])  # DESeq2 gives NA for some low-count/outlier rows

lfc = merged['log2FoldChange'].values
wt_scores = merged[['WT_rep1', 'WT_rep2']].mean(axis=1).values
# KO1_rep1 excluded — confirmed PCA outlier / likely mislabeled sample
ko_scores = merged[['KO1_rep2', 'KO2_rep1', 'KO2_rep2']].mean(axis=1).values

n_total = len(merged)
n_gained = int((lfc > 0).sum())
n_lost   = int((lfc < 0).sum())
pct_gained = 100 * n_gained / n_total
pct_lost   = 100 * n_lost / n_total
median_lfc = float(np.median(lfc))

_, mwu_p = mannwhitneyu(ko_scores, wt_scores, alternative='two-sided')
r, _ = pearsonr(wt_scores, ko_scores)

n_sig = int((merged['padj'] < 0.05).sum())
print(f"Total regions (non-NA): {n_total:,}")
print(f"Gained: {n_gained:,} ({pct_gained:.1f}%) | Lost: {n_lost:,} ({pct_lost:.1f}%)")
print(f"Median LFC: {median_lfc:+.3f} | Pearson r: {r:.3f} | MWU p: {mwu_p:.2e}")
print(f"Significant (padj<0.05): {n_sig:,}")

def fmt_p(p):
    return f'p = {p:.2e}' if p < 0.001 else f'p = {p:.3f}'

FIG_W, FIG_H, DPI = 6.693, 4.330, 300
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H), dpi=DPI)
C_WT, C_KO, C_LOST, C_NEUTRAL, C_REF = '#2166ac', '#d6604d', '#4393c3', '#333333', '#999999'

# Panel a
ax = axes[0]
parts = ax.violinplot(lfc, positions=[0], widths=0.6, showmedians=False, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor(C_WT); pc.set_edgecolor(C_NEUTRAL); pc.set_linewidth(0.75); pc.set_alpha(0.75)
q25, q50, q75 = np.percentile(lfc, [25, 50, 75])
ax.plot([0, 0], [q25, q75], color=C_NEUTRAL, lw=2.5, solid_capstyle='round', zorder=3)
ax.scatter([0], [q50], color='white', edgecolors=C_NEUTRAL, s=18, zorder=4, linewidths=0.75)
ax.axhline(0, color=C_REF, linestyle='--', lw=0.75, zorder=1)
ax.axhline(median_lfc, color=C_NEUTRAL, linestyle='-', lw=0.75, alpha=0.8, zorder=2)
ax.set_xlim(-0.6, 0.6); ax.set_xticks([])
ax.set_ylabel(r'$\log_2$ Fold Change (KO/WT)', fontsize=6, labelpad=3)
ax.tick_params(axis='y', labelsize=6, length=2.5, width=0.75, pad=2)
ax.text(0.97, 0.97, f'n = {n_total:,}\nMedian = {median_lfc:+.3f}\n{fmt_p(mwu_p)}',
        transform=ax.transAxes, va='top', ha='right', fontsize=5, color=C_NEUTRAL,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=C_REF, linewidth=0.5, alpha=0.9))
ax.set_title('Accessibility\nchange (KO vs WT)', fontsize=6.5, fontweight='bold', pad=4, color=C_NEUTRAL)
ax.text(-0.18, 1.06, 'a', transform=ax.transAxes, fontsize=8, fontweight='bold', va='top', ha='left')
sns.despine(ax=ax, offset=3, trim=True)

# Panel b
ax = axes[1]
rng = np.random.default_rng(42)
plot_idx = rng.choice(n_total, min(8000, n_total), replace=False)
ax.scatter(wt_scores[plot_idx], ko_scores[plot_idx], alpha=0.15, s=1.2, color=C_NEUTRAL, rasterized=True, linewidths=0)
max_val = max(np.percentile(wt_scores, 99.5), np.percentile(ko_scores, 99.5))
ax.plot([0, max_val], [0, max_val], color=C_REF, linestyle='--', lw=0.75, label='y = x', zorder=3)
ax.set_xlim(0, max_val); ax.set_ylim(0, max_val)
ax.text(0.05, 0.95, f'Pearson r = {r:.3f}', transform=ax.transAxes, va='top', ha='left',
        fontsize=5.5, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=C_REF, linewidth=0.5, alpha=0.9))
ax.set_xlabel('WT ATAC-seq signal (norm. counts)', fontsize=6, labelpad=3)
ax.set_ylabel('KO ATAC-seq signal (norm. counts)', fontsize=6, labelpad=3)
ax.tick_params(labelsize=6, length=2.5, width=0.75, pad=2)
ax.set_title('WT vs KO signal\ncorrelation', fontsize=6.5, fontweight='bold', pad=4, color=C_NEUTRAL)
ax.text(-0.20, 1.06, 'b', transform=ax.transAxes, fontsize=8, fontweight='bold', va='top', ha='left')
sns.despine(ax=ax, offset=3, trim=True)

# Panel c — gained/lost restricted to SIGNIFICANT peaks only (padj < 0.05)
ax = axes[2]
sig = merged[merged['padj'] < 0.05]
n_gained_sig = int((sig['log2FoldChange'] > 0).sum())
n_lost_sig   = int((sig['log2FoldChange'] < 0).sum())
n_sig_total  = len(sig)
pct_gained_sig = 100 * n_gained_sig / n_sig_total
pct_lost_sig   = 100 * n_lost_sig / n_sig_total

labels_c = ['Gained\n(more open)', 'Lost\n(more closed)']
values = [n_gained_sig, n_lost_sig]
bars = ax.bar([0, 1], values, color=[C_KO, C_LOST], edgecolor='white', linewidth=0.75, width=0.5, alpha=0.88)
for bar, val, pct in zip(bars, values, [pct_gained_sig, pct_lost_sig]):
    ax.text(bar.get_x() + bar.get_width()/2, val + max(values)*0.015, f'{val:,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=5, fontweight='bold', color=C_NEUTRAL)
ax.set_xticks([0, 1]); ax.set_xticklabels(labels_c, fontsize=6)
ax.set_ylabel('Number of significant regions\n(padj < 0.05)', fontsize=6, labelpad=3)
ax.tick_params(axis='y', labelsize=6, length=2.5, width=0.75, pad=2)
ax.tick_params(axis='x', length=0, pad=4)
ax.margins(y=0.18)
ax.text(0.5, 0.97, f'{n_sig_total:,} significant peaks\n(of {n_total:,} total)', transform=ax.transAxes, ha='center', va='top',
        fontsize=5, fontstyle='italic', color=C_NEUTRAL,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffbe6', edgecolor=C_REF, linewidth=0.5, alpha=0.9))
ax.set_title('Gained vs lost accessibility\n(significant peaks only)', fontsize=6.5, fontweight='bold', pad=4, color=C_NEUTRAL)
ax.text(-0.20, 1.06, 'c', transform=ax.transAxes, fontsize=8, fontweight='bold', va='top', ha='left')
sns.despine(ax=ax, offset=3, trim=True)

fig.suptitle('CHD8-KO chromatin accessibility changes (balanced consensus, outlier excluded)', fontsize=7, fontweight='bold', y=1.03)
plt.tight_layout(pad=0.6, w_pad=1.8, h_pad=0.5)
fig.subplots_adjust(top=0.88, left=0.10, right=0.97, bottom=0.14)

fig.savefig('/content/Figure6_balanced.tiff', dpi=DPI, bbox_inches='tight', format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig('/content/Figure6_balanced.pdf', dpi=DPI, bbox_inches='tight', format='pdf', backend='pdf')
plt.show()

from google.colab import files
files.download('/content/Figure6_balanced.tiff')
files.download('/content/Figure6_balanced.pdf')
print("Figure 6 (balanced, outlier-excluded, padj < 0.05) complete.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pybedtools
import os

# ==========================================
# 📁 1. PATHS
# ==========================================

BASE_RNA = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
TSS_PATH = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

RNA_PATHS = {
    "KO" : BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "FL" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

# ★ UPDATED — balanced, outlier-excluded ATAC results (KO1_rep1 removed, consensus rebalanced)
ATAC_DESEQ2_PATH = "/content/atac_deseq2_results_balanced.csv"

# Significance threshold for "lost"/"gained" peak calls — consistent with Fig 6 directionality
PADJ_THRESH = 0.05
LFC_THRESH  = 0.5

# ==========================================
# 🔧 2. HELPER FUNCTIONS
# ==========================================
def load_rna(path, label):
    df = pd.read_csv(path, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
    df['padj'] = df['padj'].replace(0, 1e-300)
    if 'baseMean' in df.columns:
        df = df[df['baseMean'] > 10]
    df['gene_upper'] = df['GENESYMBOL'].astype(str).str.upper().str.strip()
    print(f"✅ {label}: {len(df):,} genes loaded")
    return df

def fix_naming(feature):
    chrom = str(feature.chrom)
    if not chrom.startswith('chr'):
        feature.chrom = 'chr' + chrom
    return feature

# ==========================================
# 📥 3. LOAD ALL DATA
# ==========================================
print("=" * 60)
print("BLOCK 7: INTEGRATIVE ATAC + DOMAIN RESCUE ANALYSIS (balanced, outlier-excluded)")
print("=" * 60)

print("\n📥 Loading RNA-seq data...")
rna = {k: load_rna(v, k) for k, v in RNA_PATHS.items()}

print("\n📥 Loading balanced ATAC-seq DESeq2 results (DiffBind consensus + BAM counts)...")
atac_res = pd.read_csv(ATAC_DESEQ2_PATH).dropna(subset=['log2FoldChange'])

# region column is "chr:start-end" — parse it back into coordinates
coords = atac_res['region'].str.extract(r'(?P<chr>[^:]+):(?P<start>\d+)-(?P<end>\d+)')
atac_res['chr']   = coords['chr']
atac_res['start'] = coords['start'].astype(int)
atac_res['end']   = coords['end'].astype(int)

print(f"✅ ATAC regions loaded: {len(atac_res):,}")

# ==========================================
# 🔬 4. IDENTIFY KO-LOST ATAC PEAKS (LFC + significance, from balanced DESeq2 results)
# ==========================================
print(f"\n🔬 Identifying KO-lost/gained peaks: |LFC|>{LFC_THRESH} AND padj<{PADJ_THRESH}...")

# Now requires BOTH an effect-size threshold AND statistical significance,
# consistent with the significant-peaks-only approach used for Fig 6 directionality.
sig_atac  = atac_res[atac_res['padj'] < PADJ_THRESH].copy()
ko_lost   = sig_atac[sig_atac['log2FoldChange'] < -LFC_THRESH].copy()
ko_gained = sig_atac[sig_atac['log2FoldChange'] >  LFC_THRESH].copy()

print(f"✅ Total consensus regions        : {len(atac_res):,}")
print(f"✅ Significant regions (padj<{PADJ_THRESH}) : {len(sig_atac):,}")
print(f"✅ KO-lost peaks (LFC<-{LFC_THRESH}, sig)   : {len(ko_lost):,}")
print(f"✅ KO-gained peaks (LFC>{LFC_THRESH}, sig)  : {len(ko_gained):,}")

# ==========================================
# 🔬 5. FIND GENES NEAR KO-LOST PEAKS
# ==========================================
print("\n🔬 Finding genes near KO-lost peaks (within TSS ±2kb)...")

KO_LOST_BED = "/content/ko_lost_peaks_balanced.bed"
ko_lost[['chr', 'start', 'end']].to_csv(KO_LOST_BED, sep='\t', header=False, index=False)

if os.path.exists(TSS_PATH):
    tss_bt     = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
    ko_lost_bt = pybedtools.BedTool(KO_LOST_BED).each(fix_naming).sort()
    hits       = tss_bt.intersect(ko_lost_bt, u=True, wa=True)
    lost_genes = set([str(f[3]).upper().strip() for f in hits if len(str(f[3])) > 1])
    print(f"✅ Genes near KO-lost peaks: {len(lost_genes):,}")
else:
    print("⚠️ TSS file not found — check path")
    lost_genes = set()

# ==========================================
# 🧮 6. CALCULATE RESCUE EFFICIENCY (unchanged — RNA-seq based, not ATAC-dependent)
# ==========================================
print("\n🧮 Calculating rescue efficiency per construct...")

ko_df = rna['KO'][['gene_upper', 'log2FoldChange']].rename(columns={'log2FoldChange': 'lfc_ko'})

results = {}
for construct in ['FL', 'dC', 'dH']:
    rescue_df = rna[construct][['gene_upper', 'log2FoldChange']].rename(
        columns={'log2FoldChange': f'lfc_{construct}'})

    merged = ko_df.merge(rescue_df, on='gene_upper')

    denom = 0 - merged['lfc_ko']
    denom = denom.replace(0, np.nan)
    merged['RF'] = (merged[f'lfc_{construct}'] - merged['lfc_ko']) / denom
    merged['RF'] = merged['RF'].clip(-1, 2)

    merged['near_lost_peak'] = merged['gene_upper'].isin(lost_genes)

    results[construct] = merged
    n_near = merged['near_lost_peak'].sum()
    print(f"   {construct}: {len(merged):,} genes, {n_near:,} near KO-lost peaks")

# ==========================================
# 📊 7. STATISTICAL COMPARISON (unchanged)
# ==========================================
print("\n📊 Statistical comparison (MWU test):")
print("-" * 60)

stats_rows = []
for construct in ['FL', 'dC', 'dH']:
    df = results[construct]
    near  = df[df['near_lost_peak']]['RF'].dropna()
    other = df[~df['near_lost_peak']]['RF'].dropna()

    if len(near) > 5 and len(other) > 5:
        _, pval = mannwhitneyu(near, other, alternative='two-sided')
    else:
        pval = np.nan

    median_near  = near.median()
    median_other = other.median()

    print(f"   {construct}: near-lost median RF = {median_near:.3f} | "
          f"other median RF = {median_other:.3f} | MWU p = {pval:.2e}")

    stats_rows.append({
        'construct': construct, 'median_near': median_near, 'median_other': median_other,
        'n_near': len(near), 'n_other': len(other), 'pval': pval
    })

stats_df = pd.DataFrame(stats_rows)
print("-" * 60)

# ==========================================
# 📊 8. FIGURE
# ==========================================
import matplotlib
matplotlib.rcParams['font.family']       = 'Arial'
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5

print("\n📊 Generating Figure 7C (balanced, outlier-excluded, significant peaks only)...")

COLOR_NEAR  = '#2166AC'
COLOR_OTHER = '#B2B2B2'
FIG_WIDTH_IN, FIG_HEIGHT_IN, DPI = 6.693, 3.543, 300

fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN), dpi=DPI)
group_labels = ['Near\nlost peak', 'Other\ngenes']
construct_labels = {'FL': 'Full-length', 'dC': 'ΔChromo', 'dH': 'ΔHelicase'}
panel_letters = ['a', 'b', 'c']

for i, construct in enumerate(['FL', 'dC', 'dH']):
    ax  = axes[i]
    df  = results[construct]
    row = stats_df[stats_df['construct'] == construct].iloc[0]

    near_rf  = df[df['near_lost_peak']]['RF'].dropna()
    other_rf = df[~df['near_lost_peak']]['RF'].dropna()

    plot_data = pd.DataFrame({
        'RF': pd.concat([near_rf, other_rf]),
        'Group': (['Near\nlost peak'] * len(near_rf) + ['Other\ngenes'] * len(other_rf))
    })

    sns.violinplot(data=plot_data, x='Group', y='RF', ax=ax,
                   palette={'Near\nlost peak': COLOR_NEAR, 'Other\ngenes': COLOR_OTHER},
                   inner='quartile', linewidth=0.75, order=group_labels, saturation=0.85)

    ax.axhline(0, color='#333333', linestyle='--', lw=0.75, alpha=0.6, label='No rescue (RF=0)')
    ax.axhline(1, color='#555555', linestyle=':', lw=0.75, alpha=0.6, label='Full rescue (RF=1)')

    ax.set_ylim(-1.05, 2.05)
    ax.set_xlabel('')
    ax.set_ylabel('Rescue fraction (RF)' if i == 0 else '', fontsize=7, labelpad=3)
    ax.tick_params(axis='both', which='major', labelsize=6, length=2.5, width=0.75, pad=2)
    ax.set_xticklabels(group_labels, fontsize=6)
    ax.set_title(construct_labels[construct], fontsize=7, fontweight='bold', pad=4)
    ax.text(-0.18, 1.02, panel_letters[i], transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='top', ha='left')

    textstr = (f"n={row['n_near']} | median={row['median_near']:.2f}\n"
               f"n={row['n_other']} | median={row['median_other']:.2f}\n"
               f"MWU p={row['pval']:.1e}")
    ax.text(0.97, 0.97, textstr, transform=ax.transAxes, va='top', ha='right', fontsize=5.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#CCCCCC',
                      linewidth=0.5, alpha=0.9))

    sns.despine(ax=ax, offset=3, trim=True)

plt.tight_layout(pad=0.5, w_pad=0.8, h_pad=0.5)

PLOT_PATH_TIFF = os.path.join(SAVE_DIR, "Figure7c_ATAC_Rescue_balanced.tiff")
PLOT_PATH_PDF  = os.path.join(SAVE_DIR, "Figure7c_ATAC_Rescue_balanced.pdf")

fig.savefig(PLOT_PATH_TIFF, dpi=DPI, bbox_inches='tight', format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PLOT_PATH_PDF, dpi=DPI, bbox_inches='tight', format='pdf', backend='pdf')

size_tiff = os.path.getsize(PLOT_PATH_TIFF) / 1e6
size_pdf  = os.path.getsize(PLOT_PATH_PDF) / 1e6
print(f"\n✅ TIFF: {PLOT_PATH_TIFF} ({size_tiff:.1f} MB)")
print(f"✅ PDF:  {PLOT_PATH_PDF} ({size_pdf:.1f} MB)")
print("🎉 FIGURE 7C COMPLETE — balanced, outlier-excluded, significant peaks only")

In [ ]:
import pandas as pd
from google.colab import files

# ==========================================
# Supplementary Table S11 — balanced ATAC-seq DESeq2 results
# ==========================================
atac_res = pd.read_csv('/content/atac_deseq2_results_balanced.csv')

S11_PATH = '/content/Supplementary_Table_S11_ATAC_DESeq2_results_balanced.csv'
atac_res.to_csv(S11_PATH, index=False)
print(f"✅ S11 saved: {S11_PATH} ({len(atac_res):,} regions)")
files.download(S11_PATH)


s16_frames = []
for construct in ['FL', 'dC', 'dH']:
    df = results[construct][['gene_upper', 'lfc_ko', f'lfc_{construct}', 'RF', 'near_lost_peak']].copy()
    df = df.rename(columns={f'lfc_{construct}': 'lfc_construct'})
    df['construct'] = construct
    s16_frames.append(df)

s16 = pd.concat(s16_frames, ignore_index=True)
s16 = s16[['gene_upper', 'construct', 'lfc_ko', 'lfc_construct', 'RF', 'near_lost_peak']]

S16_PATH = '/content/Supplementary_Table_S16_rescue_fraction_balanced.csv'
s16.to_csv(S16_PATH, index=False)
print(f"✅ S16 saved: {S16_PATH} ({len(s16):,} gene-construct rows)")
files.download(S16_PATH)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_venn import venn3, venn3_circles
import pybedtools
import os
import seaborn as sns

# ==========================================
# 📁 PATHS
# ==========================================
BASE_DIR = "/content/drive/MyDrive/Chd8 data"

RNA_PATH  = os.path.join(BASE_DIR, "deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
TSS_PATH  = os.path.join(BASE_DIR, "annotations/Mus_musculus_TSS_2kb_sorted.bed")
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
SAVE_DIR  = os.path.join(BASE_DIR, "figures/")
os.makedirs(SAVE_DIR, exist_ok=True)

# ★ UPDATED — balanced, outlier-excluded ATAC results (KO1_rep1 removed, consensus rebalanced)
ATAC_DESEQ2_PATH = "/content/atac_deseq2_results_balanced.csv"

ATAC_LFC_THRESH  = 0.5
ATAC_PADJ_THRESH = 0.05   # ★ NEW — significance filter, consistent with Fig 6 / Fig 7C

# ==========================================
# 🔧 2. HELPERS
# ==========================================
def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ==========================================
# 📥 3. LOAD DATA
# ==========================================
print("=" * 60)
print("SUPPLEMENTARY FIGURE 1: MULTI-OMICS VENN (balanced, outlier-excluded ATAC)")
print("=" * 60)

print("\n📥 Loading RNA-seq...")
rna = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
rna['padj'] = rna['padj'].replace(0, 1e-300)
if 'baseMean' in rna.columns:
    rna = rna[rna['baseMean'] > 10]
rna['gene_upper'] = rna['GENESYMBOL'].astype(str).str.upper().str.strip()

degs = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'].abs() > 0.5)]['gene_upper'])
degs_up   = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'] > 0.5)]['gene_upper'])
degs_down = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'] < -0.5)]['gene_upper'])
print(f"✅ DEGs: {len(degs):,} (Up:{len(degs_up):,} Down:{len(degs_down):,})")

print("\n🔬 Loading ChIP targets...")
npc_bt   = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
esc_bt   = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
tss_bt   = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
npc_only = npc_bt.subtract(esc_bt, A=True)
hits     = tss_bt.intersect(npc_only, u=True, wa=True)
chip_targets = set([str(f[3]).upper().strip() for f in hits if len(str(f[3])) > 1])
print(f"✅ ChIP targets: {len(chip_targets):,}")

print("\n🔬 Loading balanced ATAC-seq DESeq2 results...")
atac_res = pd.read_csv(ATAC_DESEQ2_PATH).dropna(subset=['log2FoldChange'])
coords = atac_res['region'].str.extract(r'(?P<chr>[^:]+):(?P<start>\d+)-(?P<end>\d+)')
atac_res['chr']   = coords['chr']
atac_res['start'] = coords['start'].astype(int)
atac_res['end']   = coords['end'].astype(int)
print(f"✅ ATAC regions loaded: {len(atac_res):,}")

# ==========================================
# 🔬 4. ATAC "CHANGED" GENES — LFC + significance, applied once
# ==========================================
changed_atac = atac_res[(atac_res['padj'] < ATAC_PADJ_THRESH) &
                         (atac_res['log2FoldChange'].abs() > ATAC_LFC_THRESH)]
CHANGED_BED  = "/content/changed_atac_balanced.bed"
changed_atac[['chr', 'start', 'end']].to_csv(CHANGED_BED, sep='\t', header=False, index=False)

atac_bt_final   = pybedtools.BedTool(CHANGED_BED).each(fix_naming).sort()
atac_hits_final = tss_bt.intersect(atac_bt_final, u=True, wa=True)
atac_genes = set([str(f[3]).upper().strip() for f in atac_hits_final if len(str(f[3])) > 1])

print(f"\n✅ ATAC-changed regions (padj<{ATAC_PADJ_THRESH}, |LFC|>{ATAC_LFC_THRESH}): {len(changed_atac):,}")
print(f"✅ ATAC-changed genes (TSS±2kb): {len(atac_genes):,}")

# ==========================================
# 🧮 5. COMPUTE OVERLAPS
# ==========================================
A, B, C = degs, chip_targets, atac_genes

only_A = A - B - C
only_B = B - A - C
only_C = C - A - B
A_B    = (A & B) - C
A_C    = (A & C) - B
B_C    = (B & C) - A
A_B_C  = A & B & C

print(f"\n📊 OVERLAP COUNTS (balanced, outlier-excluded, significant peaks only):")
print("-" * 50)
print(f"RNA only                      : {len(only_A):,}")
print(f"ChIP only                     : {len(only_B):,}")
print(f"ATAC only                     : {len(only_C):,}")
print(f"RNA + ChIP (not ATAC)         : {len(A_B):,}")
print(f"RNA + ATAC (not ChIP)         : {len(A_C):,}")
print(f"ChIP + ATAC (not RNA)         : {len(B_C):,}")
print(f"Triple overlap (RNA+ChIP+ATAC): {len(A_B_C):,}")
print("-" * 50)

# ==========================================
# 📊 6. FIGURE
# ==========================================
fig = plt.figure(figsize=(18, 9), dpi=600)
ax1 = fig.add_subplot(1, 2, 1)

subsets = (len(only_A), len(only_B), len(A_B), len(only_C), len(A_C), len(B_C), len(A_B_C))
v = venn3(subsets=subsets, set_labels=('', '', ''),
          set_colors=('#4C72B0', '#55A868', '#C44E52'), alpha=0.55, ax=ax1)

region_map = {'100': len(only_A), '010': len(only_B), '001': len(only_C),
              '110': len(A_B), '101': len(A_C), '011': len(B_C), '111': len(A_B_C)}
for rid, val in region_map.items():
    lbl = v.get_label_by_id(rid)
    if lbl:
        lbl.set_text(f'{val:,}')
        lbl.set_fontsize(10)
        lbl.set_fontweight('bold')

if v.get_patch_by_id('111'):
    v.get_patch_by_id('111').set_color('#F39C12')
    v.get_patch_by_id('111').set_alpha(0.95)

ax1.text(-0.75, 0.60, 'RNA-seq\nDEGs', ha='center', va='center', fontsize=12,
         fontweight='bold', color='#2471A3',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#D6EAF8', alpha=0.8))
ax1.text(0.75, 0.60, 'CHD8\nChIP-seq', ha='center', va='center', fontsize=12,
         fontweight='bold', color='#1E8449',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#D5F5E3', alpha=0.8))
ax1.text(0.00, -0.72, 'ATAC-seq\nChanged', ha='center', va='center', fontsize=12,
         fontweight='bold', color='#922B21',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#FADBD8', alpha=0.8))

ax1.set_title("Supp. Fig. 1A: Multi-omics Integration\nRNA-seq DEGs | CHD8 ChIP-seq | ATAC-seq (balanced, outlier-excluded)",
              fontweight='bold', pad=20, fontsize=13)
ax1.text(0.5, -0.05,
         f"Triple overlap = {len(A_B_C):,} high-confidence targets\n"
         f"(DEG + CHD8 bound + chromatin changed, padj<{ATAC_PADJ_THRESH} & |LFC|>{ATAC_LFC_THRESH})",
         transform=ax1.transAxes, ha='center', va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='#FEF9E7', alpha=0.9))

ax2 = fig.add_subplot(1, 2, 2)
labels = [f'RNA DEGs\n(n={len(A):,})', f'CHD8 ChIP\n(n={len(B):,})', f'ATAC Changed\n(n={len(C):,})',
          f'RNA ∩ ChIP\n(n={len(A_B)+len(A_B_C):,})', f'RNA ∩ ATAC\n(n={len(A_C)+len(A_B_C):,})',
          f'ChIP ∩ ATAC\n(n={len(B_C)+len(A_B_C):,})', f'Triple ∩\n(n={len(A_B_C):,})']
values = [len(A), len(B), len(C), len(A_B)+len(A_B_C), len(A_C)+len(A_B_C), len(B_C)+len(A_B_C), len(A_B_C)]
colors = ['#4C72B0','#55A868','#C44E52','#9B59B6','#E67E22','#2ECC71','#F39C12']

bars = ax2.barh(labels[::-1], values[::-1], color=colors[::-1], edgecolor='black', alpha=0.85)
for bar, val in zip(bars, values[::-1]):
    ax2.text(bar.get_width() + max(values)*0.01, bar.get_y() + bar.get_height()/2,
              f'{val:,}', va='center', fontweight='bold', fontsize=9)
ax2.set_xlabel("Number of Genes", fontsize=12)
ax2.set_title("Supp. Fig. 1B: Set Size Summary", fontweight='bold', pad=15, fontsize=13)
ax2.set_xlim(0, max(values)*1.2)
sns.despine(ax=ax2)

plt.suptitle("Supplementary Figure 1: Multi-omics Integration Summary (balanced, outlier-excluded ATAC)",
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()

PLOT_PATH_TIFF = os.path.join(SAVE_DIR, "SuppFig1_Venn_balanced.tiff")
PLOT_PATH_PDF  = os.path.join(SAVE_DIR, "SuppFig1_Venn_balanced.pdf")

plt.savefig(PLOT_PATH_TIFF, dpi=600, bbox_inches='tight', format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
plt.savefig(PLOT_PATH_PDF, dpi=600, bbox_inches='tight', format='pdf')
plt.show()

# ==========================================
# 📋 7. SUMMARY TABLE (Supplementary Table S12)
# ==========================================
summary = pd.DataFrame({
    'Metric': ['Total DEGs', 'Upregulated', 'Downregulated', 'CHD8 ChIP targets',
               'ATAC-changed genes', 'RNA + ChIP', 'RNA + ATAC', 'ChIP + ATAC', 'Triple overlap'],
    'Value_balanced_outlier_excluded': [len(A), len(degs_up), len(degs_down), len(B), len(C),
                                         len(A_B)+len(A_B_C), len(A_C)+len(A_B_C),
                                         len(B_C)+len(A_B_C), len(A_B_C)]
})
TABLE_PATH = os.path.join(SAVE_DIR, "Supplementary_Table_S12_multiomics_overlap_genes_balanced.csv")
summary.to_csv(TABLE_PATH, index=False)

print(f"\n✅ Figure (TIFF): {PLOT_PATH_TIFF}")
print(f"✅ Figure (PDF):  {PLOT_PATH_PDF}")
print(f"✅ Table saved:   {TABLE_PATH}")
print(f"\n📝 Triple overlap (high-confidence targets) = {len(A_B_C):,}")

# ==========================================
# ⬇️ 8. DOWNLOAD FIGURE + TABLE
# ==========================================
from google.colab import files
files.download(PLOT_PATH_TIFF)
files.download(PLOT_PATH_PDF)
files.download(TABLE_PATH)

In [ ]:
import subprocess
import pandas as pd
import os

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
bam_dir  = "/content/drive/Shareddrives/Chd8/ChIP_ATAC_BAM"
peak_dir = "/content/drive/Shareddrives/Chd8/MACS2_peaks/ATAC"

MIN_OVERLAP_VALUES = [1, 2, 3, 4]     # consensus peak support threshold sweep
PADJ_VALUES        = [0.01, 0.05, 0.1]
LOG2FC_VALUES      = [0, 0.5, 1]

frag_bams_all = {
    "WT_rep1":  "/content/frag_bams/WT_rep1.frag.bam",
    "WT_rep2":  "/content/frag_bams/WT_rep2.frag.bam",
    "KO1_rep2": "/content/frag_bams/KO1_rep2.frag.bam",   # KO1_rep1 excluded (confirmed outlier)
    "KO2_rep1": "/content/frag_bams/KO2_rep1.frag.bam",
    "KO2_rep2": "/content/frag_bams/KO2_rep2.frag.bam",
}
sample_names = list(frag_bams_all.keys())   # order matters — must match bedtools multicov output columns

sweep_rows = []

for min_overlap in MIN_OVERLAP_VALUES:

    print(f"\n{'='*70}\nminOverlap = {min_overlap}\n{'='*70}")

    # -----------------------------------------------------------------
    # STEP 1: Build consensus peak set at this minOverlap
    # (consensus-building samples: 1 per KO clone, balanced, outlier excluded)
    # -----------------------------------------------------------------
    r_consensus = f'''
    library(DiffBind)

    samples_balanced <- data.frame(
      SampleID  = c("WT_rep1", "WT_rep2", "KO1_rep2", "KO2_rep1"),
      Condition = c("WT", "WT", "KO", "KO"),
      Replicate = c(1, 2, 1, 1),
      bamReads  = c(
        file.path("{bam_dir}", "ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam"),
        file.path("{bam_dir}", "ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
        file.path("{bam_dir}", "KO_1_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
        file.path("{bam_dir}", "KO_2_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam")
      ),
      Peaks = c(
        file.path("{peak_dir}", "ATAC_Diff_1/ATAC_Diff_1_peaks.narrowPeak"),
        file.path("{peak_dir}", "ATAC_Diff_2/ATAC_Diff_2_peaks.narrowPeak"),
        file.path("{peak_dir}", "KO_1_ATAC_Diff_2/KO_1_ATAC_Diff_2_peaks.narrowPeak"),
        file.path("{peak_dir}", "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1_peaks.narrowPeak")
      ),
      PeakCaller = "narrow"
    )

    atac <- dba(sampleSheet = samples_balanced)

    # Build a NEW consensus peakset at this specific minOverlap (this is the
    # step that was missing before — bRetrieve alone does not rebuild the
    # peakset, it just returns whatever consensus already existed on the object)
    atac <- dba.peakset(atac, consensus = DBA_CONDITION, minOverlap = {min_overlap})
    consensus_only <- dba(atac, mask = atac$masks$Consensus, minOverlap = 1)
    consensus <- dba.peakset(consensus_only, bRetrieve = TRUE)

    df <- as.data.frame(consensus)[, c("seqnames", "start", "end")]
    write.table(df, "/content/sweep_consensus_mo{min_overlap}.bed", sep = "\\t",
                quote = FALSE, row.names = FALSE, col.names = FALSE)
    cat("Wrote", nrow(df), "peaks at minOverlap={min_overlap}\\n")
    '''
    with open(f'/content/sweep_consensus_mo{min_overlap}.R', 'w') as f:
        f.write(r_consensus)

    proc = subprocess.run(['Rscript', f'/content/sweep_consensus_mo{min_overlap}.R'],
                           capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print("ERROR building consensus:", proc.stderr[-1500:])
        continue

    bed_path = f'/content/sweep_consensus_mo{min_overlap}.bed'
    n_peaks_consensus = sum(1 for _ in open(bed_path))
    if n_peaks_consensus == 0:
        print(f"  Skipping minOverlap={min_overlap}: 0 peaks in consensus set")
        continue

    # -----------------------------------------------------------------
    # STEP 2: Recount all 5 samples against this consensus set
    # -----------------------------------------------------------------
    counts_path = f'/content/sweep_counts_mo{min_overlap}.txt'
    cmd = ['bedtools', 'multicov', '-bams'] + [frag_bams_all[s] for s in sample_names] + \
          ['-bed', bed_path]
    with open(counts_path, 'w') as out:
        p = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        print("ERROR in bedtools multicov:", p.stderr[-1500:])
        continue

    cols = ['chr', 'start', 'end'] + sample_names
    counts = pd.read_csv(counts_path, sep='\t', header=None, names=cols)
    counts_csv = f'/content/sweep_counts_mo{min_overlap}.csv'
    counts.to_csv(counts_csv, index=False)

    # -----------------------------------------------------------------
    # STEP 3: Run DESeq2 on this count matrix
    # -----------------------------------------------------------------
    sample_cols_r = ', '.join([f'"{s}"' for s in sample_names])
    r_deseq2 = f'''
    library(DESeq2)

    counts <- read.csv("{counts_csv}")
    count_mat <- as.matrix(counts[, c({sample_cols_r})])
    rownames(count_mat) <- paste0(counts$chr, ":", counts$start, "-", counts$end)

    coldata <- data.frame(
      condition = factor(c("WT","WT","KO","KO","KO"), levels = c("WT","KO")),
      row.names = colnames(count_mat)
    )

    dds <- DESeqDataSetFromMatrix(countData = count_mat, colData = coldata, design = ~ condition)
    dds <- DESeq(dds)
    res <- results(dds, contrast = c("condition", "KO", "WT"))
    res_df <- as.data.frame(res)
    res_df$region <- rownames(res_df)

    write.csv(res_df, "/content/sweep_deseq2_mo{min_overlap}.csv", row.names = FALSE)
    cat("Saved DESeq2 results for minOverlap={min_overlap}\\n")
    '''
    with open(f'/content/sweep_deseq2_mo{min_overlap}.R', 'w') as f:
        f.write(r_deseq2)

    proc = subprocess.run(['Rscript', f'/content/sweep_deseq2_mo{min_overlap}.R'],
                           capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print("ERROR running DESeq2:", proc.stderr[-1500:])
        continue

    # -----------------------------------------------------------------
    # STEP 4: Sweep padj x log2FC on this minOverlap's DESeq2 results
    # -----------------------------------------------------------------
    res_df = pd.read_csv(f'/content/sweep_deseq2_mo{min_overlap}.csv').dropna(subset=['log2FoldChange', 'padj'])

    for padj_t in PADJ_VALUES:
        for lfc_t in LOG2FC_VALUES:
            sig = res_df[(res_df['padj'] < padj_t) & (res_df['log2FoldChange'].abs() > lfc_t)]
            n_sig = len(sig)
            n_gained = int((sig['log2FoldChange'] > 0).sum())
            n_lost   = int((sig['log2FoldChange'] < 0).sum())
            sweep_rows.append({
                'minOverlap': min_overlap,
                'n_consensus_peaks': n_peaks_consensus,
                'padj_threshold': padj_t,
                'log2fc_threshold': lfc_t,
                'n_significant': n_sig,
                'n_gained': n_gained,
                'n_lost': n_lost,
                'pct_gained': 100 * n_gained / n_sig if n_sig > 0 else None,
                'pct_lost': 100 * n_lost / n_sig if n_sig > 0 else None,
            })

sweep_df = pd.DataFrame(sweep_rows)
print("\n" + "=" * 70)
print("FULL SENSITIVITY SWEEP RESULTS")
print("=" * 70)
print(sweep_df.to_string(index=False))

SWEEP_PATH = '/content/Supplementary_Table_ATAC_parameter_sensitivity_sweep.csv'
sweep_df.to_csv(SWEEP_PATH, index=False)
print(f"\n✅ Saved: {SWEEP_PATH}")

from google.colab import files
files.download(SWEEP_PATH)

In [ ]:
!apt-get install -y bedtools -q
!pip install pybedtools -q

In [ ]:
# --------------------------------------------------------------------
# 0. MOUNT DRIVE
# --------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = "/content/drive/MyDrive/Chd8 data"
SAVE_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(SAVE_DIR, exist_ok=True)

# --------------------------------------------------------------------
# 1. INSTALL R (DiffBind, DESeq2) + samtools/bedtools + pybedtools
# --------------------------------------------------------------------
import subprocess
print("Installing R packages (this can take a few minutes)...")
subprocess.run(['wget', '-q',
    'https://raw.githubusercontent.com/eddelbuettel/r2u/master/inst/scripts/add_cranapt_jammy.sh',
    '-O', 'add_cranapt.sh'])
subprocess.run(['bash', 'add_cranapt.sh'])
subprocess.run(['apt-get', 'install', '-y', 'r-bioc-diffbind', 'r-bioc-deseq2',
                 'samtools', 'bedtools', '--quiet'])

check = subprocess.run(['Rscript', '-e',
    'cat("DiffBind:", "DiffBind" %in% rownames(installed.packages()), "\\n");'
    'cat("DESeq2:", "DESeq2" %in% rownames(installed.packages()), "\\n")'],
    capture_output=True, text=True)
print(check.stdout)

get_ipython().system('pip install pybedtools -q')
import pybedtools

# --------------------------------------------------------------------
# 2. GENERATE FRAGMENT BAMs for the 5 samples needed
#    (WT_rep1, WT_rep2, KO1_rep2, KO2_rep1, KO2_rep2 — KO1_rep1 excluded,
#    confirmed PCA outlier / likely mislabel, per Methods)
# --------------------------------------------------------------------
bam_dir = '/content/drive/Shareddrives/Chd8/ChIP_ATAC_BAM'
samples = {
    'WT_rep1':  f'{bam_dir}/ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam',
    'WT_rep2':  f'{bam_dir}/ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam',
    'KO1_rep2': f'{bam_dir}/KO_1_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam',
    'KO2_rep1': f'{bam_dir}/KO_2_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam',
    'KO2_rep2': f'{bam_dir}/KO_2_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam',
}

os.makedirs('/content/frag_bams', exist_ok=True)

for name, path in samples.items():
    out_bam = f'/content/frag_bams/{name}.frag.bam'
    print(f"Processing {name}...")
    cmd = (f'samtools view -@ 2 -b -f 0x40 -F 0x904 "{path}" > "{out_bam}" '
           f'&& samtools index "{out_bam}"')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("  ERROR:", result.stderr[-1000:])
    else:
        size_mb = os.path.getsize(out_bam) / (1024 * 1024)
        print(f"  Done -> {out_bam} ({size_mb:.1f} MB)")

# --------------------------------------------------------------------
# 3. STEP A — rebuild BALANCED consensus peak set (excludes KO1_rep1),
#    minOverlap = 2, matching Methods (4-sample consensus: 2 WT + 1/clone)
# --------------------------------------------------------------------
r_script_consensus = '''
library(DiffBind)

bam_dir  <- "/content/drive/Shareddrives/Chd8/ChIP_ATAC_BAM"
peak_dir <- "/content/drive/Shareddrives/Chd8/MACS2_peaks/ATAC"

samples_balanced <- data.frame(
  SampleID  = c("WT_rep1", "WT_rep2", "KO1_rep2", "KO2_rep1"),
  Condition = c("WT", "WT", "KO", "KO"),
  Replicate = c(1, 2, 1, 1),
  bamReads  = c(
    file.path(bam_dir, "ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_1_ATAC_Diff_2.filtered.nomt.unbl.sortcoord.bam"),
    file.path(bam_dir, "KO_2_ATAC_Diff_1.filtered.nomt.unbl.sortcoord.bam")
  ),
  Peaks = c(
    file.path(peak_dir, "ATAC_Diff_1/ATAC_Diff_1_peaks.narrowPeak"),
    file.path(peak_dir, "ATAC_Diff_2/ATAC_Diff_2_peaks.narrowPeak"),
    file.path(peak_dir, "KO_1_ATAC_Diff_2/KO_1_ATAC_Diff_2_peaks.narrowPeak"),
    file.path(peak_dir, "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1_peaks.narrowPeak")
  ),
  PeakCaller = "narrow"
)

atac_balanced <- dba(sampleSheet = samples_balanced)
print(atac_balanced)

consensus <- dba.peakset(atac_balanced, bRetrieve = TRUE, minOverlap = 2)
df <- as.data.frame(consensus)[, c("seqnames", "start", "end")]
write.table(df, "/content/consensus_peaks_balanced.bed", sep = "\\t",
            quote = FALSE, row.names = FALSE, col.names = FALSE)
cat("\\nWrote", nrow(df), "balanced consensus peaks to BED\\n")
'''
with open('/content/step_consensus_balanced.R', 'w') as f:
    f.write(r_script_consensus)

process = subprocess.Popen(['Rscript', '/content/step_consensus_balanced.R'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

# --------------------------------------------------------------------
# 4. STEP B — recount reads at balanced consensus peaks
#    (5 non-outlier samples: 2 WT + 1 KO1 rep + 2 KO2 reps)
# --------------------------------------------------------------------
frag_bams = [
    '/content/frag_bams/WT_rep1.frag.bam',
    '/content/frag_bams/WT_rep2.frag.bam',
    '/content/frag_bams/KO1_rep2.frag.bam',
    '/content/frag_bams/KO2_rep1.frag.bam',
    '/content/frag_bams/KO2_rep2.frag.bam',
]

cmd = ['bedtools', 'multicov', '-bams', *frag_bams,
       '-bed', '/content/consensus_peaks_balanced.bed']
with open('/content/atac_counts_balanced.txt', 'w') as out:
    process = subprocess.Popen(cmd, stdout=out, stderr=subprocess.PIPE, text=True)
    _, err = process.communicate()

print("Exit code:", process.returncode)
if err:
    print("STDERR:", err[-2000:])

# --------------------------------------------------------------------
# 5. STEP C — label counts, save
# --------------------------------------------------------------------
import pandas as pd

cols = ['chr', 'start', 'end', 'WT_rep1', 'WT_rep2', 'KO1_rep2', 'KO2_rep1', 'KO2_rep2']
counts = pd.read_csv('/content/atac_counts_balanced.txt', sep='\t', header=None, names=cols)
print("Shape:", counts.shape)
counts.to_csv('/content/atac_counts_balanced_labeled.csv', index=False)

# --------------------------------------------------------------------
# 6. STEP D — DESeq2 on the balanced, outlier-excluded dataset
# --------------------------------------------------------------------
ATAC_DESEQ2_PATH = os.path.join(BASE_DIR, "atac_deseq2_results_balanced.csv")

r_script_deseq2 = f'''
library(DESeq2)

counts <- read.csv("/content/atac_counts_balanced_labeled.csv")

count_mat <- as.matrix(counts[, c("WT_rep1","WT_rep2","KO1_rep2","KO2_rep1","KO2_rep2")])
rownames(count_mat) <- paste0(counts$chr, ":", counts$start, "-", counts$end)

coldata <- data.frame(
  condition = factor(c("WT","WT","KO","KO","KO"), levels = c("WT","KO")),
  clone     = factor(c("WT","WT","KO1","KO2","KO2")),
  row.names = colnames(count_mat)
)

dds <- DESeqDataSetFromMatrix(countData = count_mat, colData = coldata, design = ~ condition)
dds <- DESeq(dds)

res <- results(dds, contrast = c("condition", "KO", "WT"))
res_df <- as.data.frame(res)
res_df$region <- rownames(res_df)

cat("DESeq2 summary (balanced, outlier excluded):\\n")
summary(res)

write.csv(res_df, "{ATAC_DESEQ2_PATH}", row.names = FALSE)
cat("Saved to {ATAC_DESEQ2_PATH}\\n")
'''
with open('/content/run_deseq2_balanced.R', 'w') as f:
    f.write(r_script_deseq2)

process = subprocess.Popen(['Rscript', '/content/run_deseq2_balanced.R'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

print(f"\n✅ Balanced ATAC DESeq2 results now saved permanently to Drive:\n   {ATAC_DESEQ2_PATH}")

# --------------------------------------------------------------------
# 7. TRIPLE-OVERLAP EXTRACTION (RNA DEGs ∩ CHD8 ChIP targets ∩ ATAC-changed)
# --------------------------------------------------------------------
RNA_PATH  = os.path.join(BASE_DIR, "deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
TSS_PATH  = os.path.join(BASE_DIR, "annotations/Mus_musculus_TSS_2kb_sorted.bed")
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"

ATAC_LFC_THRESH  = 0.5
ATAC_PADJ_THRESH = 0.05

def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

print("\n📥 Loading RNA-seq...")
rna = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
rna['padj'] = rna['padj'].replace(0, 1e-300)
if 'baseMean' in rna.columns:
    rna = rna[rna['baseMean'] > 10]
rna['gene_upper'] = rna['GENESYMBOL'].astype(str).str.upper().str.strip()

degs = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'].abs() > 0.5)]['gene_upper'])
print(f"✅ DEGs: {len(degs):,}  (should be 2,752)")

print("\n🔬 Loading ChIP targets...")
npc_bt   = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
esc_bt   = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
tss_bt   = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
npc_only = npc_bt.subtract(esc_bt, A=True)
hits     = tss_bt.intersect(npc_only, u=True, wa=True)
chip_targets = set([str(f[3]).upper().strip() for f in hits if len(str(f[3])) > 1])
print(f"✅ ChIP targets: {len(chip_targets):,}  (should be 3,754)")

print("\n🔬 Loading balanced ATAC-seq DESeq2 results...")
atac_res = pd.read_csv(ATAC_DESEQ2_PATH).dropna(subset=['log2FoldChange'])
coords = atac_res['region'].str.extract(r'(?P<chr>[^:]+):(?P<start>\d+)-(?P<end>\d+)')
atac_res['chr']   = coords['chr']
atac_res['start'] = coords['start'].astype(int)
atac_res['end']   = coords['end'].astype(int)

changed_atac = atac_res[(atac_res['padj'] < ATAC_PADJ_THRESH) &
                         (atac_res['log2FoldChange'].abs() > ATAC_LFC_THRESH)]
CHANGED_BED  = "/content/changed_atac_balanced.bed"
changed_atac[['chr', 'start', 'end']].to_csv(CHANGED_BED, sep='\t', header=False, index=False)

atac_bt_final   = pybedtools.BedTool(CHANGED_BED).each(fix_naming).sort()
atac_hits_final = tss_bt.intersect(atac_bt_final, u=True, wa=True)
atac_genes = set([str(f[3]).upper().strip() for f in atac_hits_final if len(str(f[3])) > 1])

print(f"✅ ATAC-changed regions: {len(changed_atac):,}  (should be 4,486)")
print(f"✅ ATAC-changed genes (TSS±2kb): {len(atac_genes):,}  (should be 473)")

# --------------------------------------------------------------------
# 8. 🎯 FINAL TRIPLE OVERLAP — the 8 gene names
# --------------------------------------------------------------------
triple_overlap = chip_targets & degs & atac_genes
print(f"\n🎯 TRIPLE-OVERLAP GENES: {len(triple_overlap)}  (should be 8)")
print(sorted(triple_overlap))

triple_df = rna[rna['gene_upper'].isin(triple_overlap)][
    ['GENESYMBOL', 'gene_upper', 'log2FoldChange', 'padj', 'baseMean']
].drop_duplicates().sort_values('log2FoldChange')

OUT_PATH = os.path.join(SAVE_DIR, "triple_overlap_genes_FINAL_8.csv")
triple_df.to_csv(OUT_PATH, index=False)
print(f"\n📋 Saved final 8-gene table to: {OUT_PATH}")
print(triple_df.to_string(index=False))